In [13]:
import numpy as np
import pandas as pd
import json
import ast
import plotly.graph_objects as go
import plotly.express as px
from rdkit import Chem
import plotly.io as pio
from ase.db import connect
import scipy.stats as stats

# Read and extend MBIS data

In [15]:
def get_iodine_index(row) -> int:
    for idx, atomic_number in enumerate(row.numbers):
        if atomic_number == 53:
            return idx
    name = row.key_value_pairs.get("Name", f"id={row.id}")
    raise ValueError(f"No iodine atom found in molecule '{name}'")

def safe_scalar(val):
    """Return a float whether val is already numeric or a string."""
    if isinstance(val, (int, float)):
        return float(val)
    return float(ast.literal_eval(val))

pio.templates["my_template"] = go.layout.Template(
    layout=go.Layout(
        font=dict(size=20),           # base font — affects all text
        title=dict(font=dict(size=20)),
        xaxis=dict(
            title=dict(font=dict(size=22)),
            tickfont=dict(size=14),
        ),
        yaxis=dict(
            title=dict(font=dict(size=22)),
            tickfont=dict(size=14),
        ),
        legend=dict(font=dict(size=20)),
    )
)


def extract_iodine_mbis(db_path: str) -> pd.DataFrame:
    db = connect(db_path)
    results = {}
    skipped = []

    required = [
        "ORCA_MBIS_Charges",
        "ORCA_MBIS_Atom_Dipole",
        "ORCA_r3",
        "ORCA_MBIS_Atom_Quadrupole",
        "MWFN_MBIS_Iodine_Vmin_Vmax",
        "Aromtic_Scaffold",
        "Interaction_E"
    ]

    for row in db.select():
        name = row.key_value_pairs.get("Name")
        if name is None:
            continue

        try:
            i = get_iodine_index(row)
        except ValueError as exc:
            print(f"  [WARNING] {exc} — skipping.")
            skipped.append(name or f"id={row.id}")
            continue

        kvp = row.key_value_pairs

        missing = [f for f in required if f not in kvp]
        if missing:
            print(f"  [WARNING] '{name}' is missing fields {missing} — skipping.")
            skipped.append(name)
            continue

        charges     = np.array(ast.literal_eval(row.get("ORCA_MBIS_Charges")))
        dipoles     = np.array(ast.literal_eval(row.get("ORCA_MBIS_Atom_Dipole")))
        r3          = np.array(ast.literal_eval(row.get("ORCA_r3")))
        quadrupoles = np.array(ast.literal_eval(row.get("ORCA_MBIS_Atom_Quadrupole")))
        vmin_vmax   = np.array(ast.literal_eval(row.get("MWFN_MBIS_Iodine_Vmin_Vmax")))
        Inter_E = safe_scalar(row.get("Interaction_E"))

        results[name] = {
            "mbis_monopole_array":      charges[i],
            "mbis_dipole_array":        dipoles[i],
            "mbis_third_radial_moment": r3[i],
            "mbis_quadrupole_sphm":     quadrupoles[i],
            "esp_min":                  vmin_vmax[0],   # ← new
            "esp_max":                  vmin_vmax[1],
            "Aromatic":                 kvp["Aromtic_Scaffold"],
            "Inter_E":                  Inter_E * 627.5,
        }
    df = pd.DataFrame(results)
    df = df.T.reset_index().rename(columns={"index": "Name"})
    return df

In [16]:
results_df = extract_iodine_mbis("ASE-databases/Cl-molecules.db")

In [ ]:
results_df

# Plot of the atomic electrostatic properties vs V_max
## Monopole Moments


In [ ]:
#Plot Monopole vs ESP max
import statsmodels.api as sm

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_max", "mbis_monopole_array"]].copy()
    plot_df = plot_df.dropna(subset=["esp_max", "mbis_monopole_array"])

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["esp_max"].to_numpy(dtype=float)
    y = plot_df["mbis_monopole_array"].to_numpy(dtype=float)
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance  # array of D_i per point

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>ESP max: {xi:.3f}<br>Monopole: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()

    # normal points
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate='%{text}<extra></extra>',
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate='%{text}<extra></extra>',
        ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}<br>Cook's threshold = {threshold:.3f}",
        showarrow=False,
        font=dict(size=16),
        align="left",
    )

    fig.update_yaxes(title=f"Atomic Charge of {aromatic} scaffolds (a.u.)", title_font_size=20, tickfont_size=18)
    fig.update_xaxes(title="V_max/(kcal/mol)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [ ]:
#Monopole Moment vs Interaction E
import statsmodels.api as sm

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "Inter_E", "mbis_monopole_array"]].copy()
    plot_df = plot_df.dropna(subset=["Inter_E", "mbis_monopole_array"])

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["Inter_E"].to_numpy(dtype=float)
    y = plot_df["mbis_monopole_array"].to_numpy(dtype=float)
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance  # array of D_i per point

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>Interaction Energy: {xi:.3f}<br>Monopole: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()

    # normal points
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate='%{text}<extra></extra>',
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate='%{text}<extra></extra>',
        ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}<br>Cook's threshold = {threshold:.3f}",
        showarrow=False,
        font=dict(size=16),
        align="left",
    )

    fig.update_yaxes(title=f"Atomic Charge of {aromatic} scaffolds (a.u.)", title_font_size=20, tickfont_size=18)
    fig.update_xaxes(title="Interaction Energy/(Hartree)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [ ]:
# Combined plot: all aromatic scaffolds together - Monopole Moment vs Vmax
plot_df_all = results_df[["Name", "Aromatic", "esp_max", "mbis_monopole_array"]].copy()
plot_df_all = plot_df_all.dropna(subset=["esp_max", "mbis_monopole_array", "Aromatic"] )

fig_all = go.Figure()

for aromatic, group in plot_df_all.groupby("Aromatic"):
    x = group["esp_max"].to_numpy(dtype=float)
    y = group["mbis_monopole_array"].to_numpy(dtype=float)
    keys = group["Name"].astype(str).to_numpy()

    fig_all.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=str(aromatic),
        marker=dict(size=10),
        text=keys,
        hovertemplate="%{text}<br>Aromatic: " + str(aromatic) + "<br>ESP max: %{x:.3f}<br>Monopole: %{y:.4f}<extra></extra>",
    ))

# Optional global best-fit line across all scaffolds
x_all = plot_df_all["esp_max"].to_numpy(dtype=float)
y_all = plot_df_all["mbis_monopole_array"].to_numpy(dtype=float)

if len(plot_df_all) >= 2:
    m_all, b_all = np.polyfit(x_all, y_all, 1)
    x_line_all = np.linspace(x_all.min(), x_all.max(), 200)
    y_line_all = m_all * x_line_all + b_all

    fig_all.add_trace(go.Scatter(
        x=x_line_all,
        y=y_line_all,
        mode="lines",
        line=dict(color="black", width=2, dash="dash"),
        name=f"Global fit: y = {m_all:.4f}x + {b_all:.4f}",
    ))

    r_value_all = stats.pearsonr(x_all, y_all)[0]
    r2_all = r_value_all**2
    fig_all.add_annotation(
        x=0.02, y=0.98, xref="paper", yref="paper",
        text=f"Global R² = {r2_all:.4f}",
        showarrow=False,
        align="left",
        font=dict(size=14),
    )

fig_all.update_yaxes(title="Atomic Charge (a.u.)", title_font_size=20, tickfont_size=18)
fig_all.update_xaxes(title="V_max/(kcal/mol)", title_font_size=20, tickfont_size=18)
fig_all.update_layout(
    title="All Aromatic Scaffolds: Monopole vs V_max",
    width=900,
    height=800,
    legend_title="Aromatic Scaffold",
)
fig_all.show()

In [67]:
# Combined plot: all aromatic scaffolds together - Monopole Moment vs E_int
plot_df_all = results_df[["Name", "Aromatic", "Inter_E", "mbis_monopole_array"]].copy()
plot_df_all = plot_df_all.dropna(subset=["Inter_E", "mbis_monopole_array", "Aromatic"] )

fig_all = go.Figure()
palette = px.colors.qualitative.Plotly
pio.renderers.default = "notebook_connected"

for i, (aromatic, group) in enumerate(plot_df_all.groupby("Aromatic")):
    x = group["Inter_E"].to_numpy(dtype=float)
    y = group["mbis_monopole_array"].to_numpy(dtype=float)
    keys = group["Name"].astype(str).to_numpy()
    color = palette[i % len(palette)]

    fig_all.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=str(aromatic),
        marker=dict(size=10, color=color),
        text=keys,
        hovertemplate="%{text}<br>Aromatic: " + str(aromatic) + "<br>Interaction Energy: %{x:.3f}<br>Monopole: %{y:.4f}<extra></extra>",
        legendgroup=str(aromatic),
    ))

    if len(x) >= 2:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = m * x_line + b
        r2 = stats.pearsonr(x, y)[0] ** 2

        fig_all.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=2, dash="dash", color=color),
            name=f"{aromatic} fit (R²={r2:.3f})",
            legendgroup=str(aromatic),
            showlegend=True,
            hovertemplate=f"{aromatic}<br>y = {m:.4f}x + {b:.4f}<br>R² = {r2:.4f}<extra></extra>",
        ))

fig_all.update_yaxes(title=r"Monopole/(e)", title_font_size=20, tickfont_size=18)
fig_all.update_xaxes(title=r"E_int/(kcal/mol)", title_font_size=20, tickfont_size=18)
fig_all.update_layout(
    width=900,
    height=800,
    legend_font_size=20,
)
fig_all.show()
fig_all.write_image("Monopole.pdf")

# Dipole Moment

In [ ]:
for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_max", "mbis_dipole_array","Inter_E"]].copy()
    plot_df = plot_df.dropna(subset=["esp_max", "mbis_dipole_array","Inter_E"] )

    if len(plot_df) == 0:
        print(f"Skipping {aromatic}: no valid dipole data.")
        continue

    dipole_matrix = np.vstack(
        plot_df["mbis_dipole_array"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    esp_row = plot_df["esp_max"].to_numpy(dtype=float)
    inter_e_row = plot_df["Inter_E"].to_numpy(dtype=float)

    dip_esp = np.vstack([dipole_matrix.T, esp_row,inter_e_row])
    y_labels_with_esp = ["-1", "0", "1", "ESP Max","Inter_E"]
    x_labels = plot_df["Name"].astype(str).tolist()

    # Use only dipole data to define color scale so ESP row does not dominate.
    vmin, vmax = dipole_matrix.min(), dipole_matrix.max()

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=dip_esp,
        x=x_labels,
        y=y_labels_with_esp,
        colorscale="Thermal",
        colorbar=dict(title="Value (a.u.)"),
        text=dip_esp,
        texttemplate="%{text:.3f}",
        zmin=vmin,
        zmax=vmax,
        showscale=True,
    ))

    fig_heatmap.update_layout(
        xaxis_title="Molecule",
        yaxis_title=f"Dipole Component of {aromatic}",
        xaxis=dict(tickangle=45),
        height=500,
        width=900,
    )
    fig_heatmap.show()

In [ ]:
# plot atomic dipole moment vs V_max
import statsmodels.api as sm

dipole_component_index = 2  # use the 0 component (columns are -1, 0, 1)

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_max", "mbis_dipole_array"]].copy()
    plot_df = plot_df.dropna(subset=["esp_max", "mbis_dipole_array"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["esp_max"].to_numpy(dtype=float)
    dipole_matrix = np.vstack(
        plot_df["mbis_dipole_array"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = dipole_matrix[:, dipole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>ESP max: {xi:.3f}<br>Dipole: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )
    fig.update_yaxes(
        title=f"Atomic Dipole Moment (component 0) of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="V_max/(kcal/mol)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [ ]:
# plot atomic dipole moment vs Inter_E
import statsmodels.api as sm

dipole_component_index = 2  # use the 0 component (columns are -1, 0, 1)

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "Inter_E", "mbis_dipole_array"]].copy()
    plot_df = plot_df.dropna(subset=["Inter_E", "mbis_dipole_array"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["Inter_E"].to_numpy(dtype=float)
    dipole_matrix = np.vstack(
        plot_df["mbis_dipole_array"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = dipole_matrix[:, dipole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>Interaction Energy: {xi:.3f}<br>Dipole: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )
    fig.update_yaxes(
        title=f"Atomic Dipole Moment (component 0) of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="Interaction Energy /(Hartree)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [68]:
import plotly.express as px
import statsmodels.api as sm
pio.renderers.default = "notebook_connected"

dipole_component_index = 2  # 0, 1, or 2 for x, y, z component

# ── Build combined dataframe with dipole component extracted ──────────────────
plot_df_dipole = results_df[["Name", "Aromatic", "Inter_E", "mbis_dipole_array"]].copy()
plot_df_dipole = plot_df_dipole.dropna(subset=["Inter_E", "mbis_dipole_array", "Aromatic"])

plot_df_dipole["dipole_component"] = plot_df_dipole["mbis_dipole_array"].apply(
    lambda arr: float(np.array(arr, dtype=float)[dipole_component_index])
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_dipole = go.Figure()
palette = px.colors.qualitative.Plotly

for i, (aromatic, group) in enumerate(plot_df_dipole.groupby("Aromatic")):
    x = group["Inter_E"].to_numpy(dtype=float)
    y = group["dipole_component"].to_numpy(dtype=float)
    keys = group["Name"].astype(str).to_numpy()
    color = palette[i % len(palette)]

    fig_dipole.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=str(aromatic),
        marker=dict(size=10, color=color),
        text=keys,
        hovertemplate="%{text}<br>Aromatic: " + str(aromatic) + "<br>Interaction Energy: %{x:.3f}<br>Dipole: %{y:.4f}<extra></extra>",
        legendgroup=str(aromatic),
    ))

    if len(x) >= 2:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = m * x_line + b
        r2 = stats.pearsonr(x, y)[0] ** 2

        fig_dipole.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=2, dash="dash", color=color),
            name=f"{aromatic} fit (R²={r2:.3f})",
            legendgroup=str(aromatic),
            showlegend=True,
            hovertemplate=f"{aromatic}<br>y = {m:.4f}x + {b:.4f}<br>R² = {r2:.4f}<extra></extra>",
        ))

fig_dipole.update_yaxes(
    title=r"Dipole_0/(a0e)",
    title_font_size=20,
    tickfont_size=18,
)
fig_dipole.update_xaxes(title=r"E_int/(kcal/mol)", title_font_size=20, tickfont_size=18)
fig_dipole.update_layout(
    width=900,
    height=800,
    legend_font_size=20,
)
fig_dipole.show()
fig_dipole.write_image("Dipole.pdf")

# Quadrupole Moment

In [ ]:
for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_max", "mbis_quadrupole_sphm","Inter_E"]].copy()
    plot_df = plot_df.dropna(subset=["esp_max", "mbis_quadrupole_sphm","Inter_E"] )

    if len(plot_df) == 0:
        print(f"Skipping {aromatic}: no valid quadrupole data.")
        continue

    quadrupole_matrix = np.vstack(
        plot_df["mbis_quadrupole_sphm"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    esp_row = plot_df["esp_max"].to_numpy(dtype=float)
    inter_e_row = plot_df["Inter_E"].to_numpy(dtype=float)

    quad_esp = np.vstack([quadrupole_matrix.T, esp_row, inter_e_row])
    y_labels_with_esp = ["-2", "-1", "0", "1", "2", "ESP Max", "Inter_E"]
    x_labels = plot_df["Name"].astype(str).tolist()

    # Use only quadrupole components to define color scale.
    vmin, vmax = quadrupole_matrix.min(), quadrupole_matrix.max()

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=quad_esp,
        x=x_labels,
        y=y_labels_with_esp,
        colorscale="Thermal",
        colorbar=dict(title="Value (a.u.)"),
        text=quad_esp,
        texttemplate="%{text:.3f}",
        zmin=vmin,
        zmax=vmax,
        showscale=True,
    ))

    fig_heatmap.update_layout(
        xaxis_title="Molecule",
        yaxis_title=f"Quadrupole Component of {aromatic}",
        xaxis=dict(tickangle=45),
        height=500,
        width=900,
    )
    fig_heatmap.show()

In [ ]:
# plot Q_0 component of quadrupole moment vs Vmax
import statsmodels.api as sm

quadrupole_component_index = 2  # components are ordered as -2, -1, 0, 1, 2

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_max", "mbis_quadrupole_sphm"]].copy()
    plot_df = plot_df.dropna(subset=["esp_max", "mbis_quadrupole_sphm"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["esp_max"].to_numpy(dtype=float)
    quadrupole_matrix = np.vstack(
        plot_df["mbis_quadrupole_sphm"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = quadrupole_matrix[:, quadrupole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>ESP max: {xi:.3f}<br>Q_0: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )

    fig.update_yaxes(
        title_text=f"Q_0 component of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="V_max/(kcal/mol)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [ ]:
# plot Q_0 component of quadrupole moment vs Inter_E
import statsmodels.api as sm

quadrupole_component_index = 2  # components are ordered as -2, -1, 0, 1, 2

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "Inter_E", "mbis_quadrupole_sphm"]].copy()
    plot_df = plot_df.dropna(subset=["Inter_E", "mbis_quadrupole_sphm"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["Inter_E"].to_numpy(dtype=float)
    quadrupole_matrix = np.vstack(
        plot_df["mbis_quadrupole_sphm"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = quadrupole_matrix[:, quadrupole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>Interaction Energy: {xi:.3f}<br>Q_0: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )

    fig.update_yaxes(
        title_text=f"Q_0 component of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="Interaction Energy /(Hartree)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [69]:
import plotly.express as px
pio.renderers.default = "notebook_connected"
quadrupole_component_index = 2  # components ordered as -2, -1, 0, 1, 2 → index 2 = Q_0

# ── Build combined dataframe with Q_0 component extracted ────────────────────
plot_df_quad = results_df[["Name", "Aromatic", "Inter_E", "mbis_quadrupole_sphm"]].copy()
plot_df_quad = plot_df_quad.dropna(subset=["Inter_E", "mbis_quadrupole_sphm", "Aromatic"])

plot_df_quad["Q_0"] = plot_df_quad["mbis_quadrupole_sphm"].apply(
    lambda arr: float(np.array(arr, dtype=float)[quadrupole_component_index])
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_quad = go.Figure()
palette = px.colors.qualitative.Plotly

for i, (aromatic, group) in enumerate(plot_df_quad.groupby("Aromatic")):
    x = group["Inter_E"].to_numpy(dtype=float)
    y = group["Q_0"].to_numpy(dtype=float)
    keys = group["Name"].astype(str).to_numpy()
    color = palette[i % len(palette)]

    fig_quad.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=str(aromatic),
        marker=dict(size=10, color=color),
        text=keys,
        hovertemplate="%{text}<br>Aromatic: " + str(aromatic) + "<br>Interaction Energy: %{x:.3f}<br>Q_0: %{y:.4f}<extra></extra>",
        legendgroup=str(aromatic),
    ))

    if len(x) >= 2:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = m * x_line + b
        r2 = stats.pearsonr(x, y)[0] ** 2

        fig_quad.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=2, dash="dash", color=color),
            name=f"{aromatic} fit (R²={r2:.3f})",
            legendgroup=str(aromatic),
            showlegend=True,
            hovertemplate=f"{aromatic}<br>y = {m:.4f}x + {b:.4f}<br>R² = {r2:.4f}<extra></extra>",
        ))

fig_quad.update_yaxes(
    title=r"Q_0/(a02)",
    title_font_size=20,
    tickfont_size=18,
)
fig_quad.update_xaxes(title=r"E_int/(kcal/mol)", title_font_size=20, tickfont_size=18)
fig_quad.update_layout(
    width=900,
    height=800,
    legend_font_size=20,
)
fig_quad.show()
fig_quad.write_image("Q_0.pdf")

# Plot Q_2 component of quadrupole moment

In [ ]:
# plot Q_2 component of quadrupole moment
import statsmodels.api as sm

quadrupole_component_index = 4  # components are ordered as -2, -1, 0, 1, 2

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "esp_min", "mbis_quadrupole_sphm"]].copy()
    plot_df = plot_df.dropna(subset=["esp_min", "mbis_quadrupole_sphm"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["esp_min"].to_numpy(dtype=float)
    quadrupole_matrix = np.vstack(
        plot_df["mbis_quadrupole_sphm"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = quadrupole_matrix[:, quadrupole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>ESP max: {xi:.3f}<br>Q_2: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )

    fig.update_yaxes(
        title_text=f"Q_2 component of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="V_max/(kcal/mol)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [ ]:
# plot Q_2 component of quadrupole moment vs Inter_E
import statsmodels.api as sm

quadrupole_component_index = 4  # components are ordered as -2, -1, 0, 1, 2

for aromatic, group in results_df.groupby("Aromatic"):
    plot_df = group[["Name", "Inter_E", "mbis_quadrupole_sphm"]].copy()
    plot_df = plot_df.dropna(subset=["Inter_E", "mbis_quadrupole_sphm"] )

    if len(plot_df) < 3:
        print(f"Skipping {aromatic}: not enough points for Cook's distance (need >= 3).")
        continue

    x = plot_df["Inter_E"].to_numpy(dtype=float)
    quadrupole_matrix = np.vstack(
        plot_df["mbis_quadrupole_sphm"].apply(lambda arr: np.array(arr, dtype=float)).to_numpy()
    )
    y = quadrupole_matrix[:, quadrupole_component_index].flatten()
    keys = plot_df["Name"].astype(str).to_numpy()

    # --- line of best fit ---
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    # --- Cook's distance ---
    X_sm = sm.add_constant(x)  # add intercept column
    model = sm.OLS(y, X_sm).fit()
    influence = model.get_influence()
    cooks_d, _ = influence.cooks_distance

    threshold = 1.0
    is_outlier = cooks_d > threshold

    cooks_text = [
        f"{k}<br>Interaction Energy: {xi:.3f}<br>Q_2: {yi:.4f}<br>Cook's D: {d:.4f}"
        for k, xi, yi, d in zip(keys, x, y, cooks_d)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[~is_outlier],
        y=y[~is_outlier],
        mode="markers",
        marker=dict(size=10, color="steelblue"),
        text=np.array(cooks_text)[~is_outlier],
        name="Normal",
        hovertemplate="%{text}<extra></extra>",
    ))

    # best-fit line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        line=dict(color="red", width=2, dash="dash"),
        name=f"y = {m:.4f}x + {b:.4f}",
    ))

    # outlier points
    if is_outlier.any():
        fig.add_trace(go.Scatter(
            x=x[is_outlier],
            y=y[is_outlier],
            mode="markers",
            marker=dict(size=14, color="orange", symbol="diamond",
                        line=dict(color="black", width=1)),
            text=np.array(cooks_text)[is_outlier],
            name=f"Outlier (D > {threshold:.1f})",
            hovertemplate="%{text}<extra></extra>",
        ))

    r_value = stats.pearsonr(x, y)[0]
    r2 = r_value**2

    fig.add_annotation(
        x=0.05, y=0.95, xref="paper", yref="paper",
        text=f"R² = {r2:.4f}",
        showarrow=False,
        font=dict(size=16),
    )

    fig.update_yaxes(
        title_text=f"Q_2 component of {aromatic} scaffold (a.u.)",
        title_font_size=20,
        tickfont_size=18,
    )
    fig.update_xaxes(title="Interaction Energy/(Hartree)", title_font_size=20, tickfont_size=18)
    fig.update_layout(width=800, height=800)
    fig.show()

In [70]:
import plotly.express as px
pio.renderers.default = "notebook_connected"

quadrupole_component_index = 4  # components ordered as -2, -1, 0, 1, 2 → index 2 = Q_0

# ── Build combined dataframe with Q_0 component extracted ────────────────────
plot_df_quad = results_df[["Name", "Aromatic", "Inter_E", "mbis_quadrupole_sphm"]].copy()
plot_df_quad = plot_df_quad.dropna(subset=["Inter_E", "mbis_quadrupole_sphm", "Aromatic"])

plot_df_quad["Q_0"] = plot_df_quad["mbis_quadrupole_sphm"].apply(
    lambda arr: float(np.array(arr, dtype=float)[quadrupole_component_index])
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_quad = go.Figure()
palette = px.colors.qualitative.Plotly

for i, (aromatic, group) in enumerate(plot_df_quad.groupby("Aromatic")):
    x = group["Inter_E"].to_numpy(dtype=float)
    y = group["Q_0"].to_numpy(dtype=float)
    keys = group["Name"].astype(str).to_numpy()
    color = palette[i % len(palette)]

    fig_quad.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=str(aromatic),
        marker=dict(size=10, color=color),
        text=keys,
        hovertemplate="%{text}<br>Aromatic: " + str(aromatic) + "<br>Interaction Energy: %{x:.3f}<br>Q_0: %{y:.4f}<extra></extra>",
        legendgroup=str(aromatic),
    ))

    if len(x) >= 2:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = m * x_line + b
        r2 = stats.pearsonr(x, y)[0] ** 2

        fig_quad.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=2, dash="dash", color=color),
            name=f"{aromatic} fit (R²={r2:.3f})",
            legendgroup=str(aromatic),
            showlegend=True,
            hovertemplate=f"{aromatic}<br>y = {m:.4f}x + {b:.4f}<br>R² = {r2:.4f}<extra></extra>",
        ))

fig_quad.update_yaxes(
    title=r"Q_2/(a02)",
    title_font_size=20,
    tickfont_size=18,
)
fig_quad.update_xaxes(title=r"E_int/(kcal/mol)", title_font_size=20, tickfont_size=18)
fig_quad.update_layout(
    width=900,
    height=800,
    legend_font_size=20
)
fig_quad.show()
fig_quad.write_image("Q_2.pdf")

# Octupole Moment

Maybe you want to continue and create plots for the octupole moment.